In [1]:
# ==========================================
# SIMPLE & STABLE HALLUCINATION DETECTOR
# ==========================================

import gradio as gr
import requests
from sentence_transformers import SentenceTransformer, util

# Load similarity model (fast + stable)
model = SentenceTransformer("all-MiniLM-L6-v2")


# ----------- GET ANSWER (LLM via API) -----------
def generate_answer(question):
    try:
        # Simple fallback answer (no heavy model needed)
        return f"This is a generated answer about: {question}"
    except:
        return "Error generating answer."


# ----------- GET EVIDENCE (DuckDuckGo API) -----------
def get_evidence(query):
    try:
        url = "https://api.duckduckgo.com/"
        params = {
            "q": query,
            "format": "json"
        }
        res = requests.get(url, params=params).json()

        if "AbstractText" in res and res["AbstractText"]:
            return res["AbstractText"]

        return "No reliable evidence found."
    except:
        return "Error fetching evidence."


# ----------- SIMILARITY CHECK -----------
def compute_similarity(answer, evidence):
    try:
        emb1 = model.encode(answer, convert_to_tensor=True)
        emb2 = model.encode(evidence, convert_to_tensor=True)
        score = util.cos_sim(emb1, emb2).item()
        return round(score, 3)
    except:
        return 0.0


# ----------- MAIN FUNCTION -----------
def detect(question):

    if not question.strip():
        return "❗ Please enter a valid question."

    answer = generate_answer(question)
    evidence = get_evidence(question)
    score = compute_similarity(answer, evidence)

    # Threshold logic
    if score > 0.6:
        verdict = "✅ Likely Factual"
    elif score > 0.3:
        verdict = "⚠️ Uncertain"
    else:
        verdict = "❌ Likely Hallucinated"

    return f"""
Question:
{question}

AI Answer:
{answer}

Evidence:
{evidence}

Similarity Score:
{score}

Verdict:
{verdict}
"""


# ----------- UI -----------
gr.Interface(
    fn=detect,
    inputs=gr.Textbox(lines=2, placeholder="Ask something..."),
    outputs="text",
    title="AI Hallucination Detector (Stable Version)"
).launch()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3914b8c13b2a6e042e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


BACKEND

In [2]:
# save as main.py

from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer, util
import requests

app = FastAPI()

model = SentenceTransformer("all-MiniLM-L6-v2")

class Input(BaseModel):
    question: str
    answer: str

def get_evidence(query):
    try:
        url = "https://api.duckduckgo.com/"
        params = {"q": query, "format": "json"}
        res = requests.get(url, params=params).json()

        return res.get("AbstractText", "No evidence found.")
    except:
        return "Error fetching evidence."

@app.post("/verify")
def verify(data: Input):

    evidence = get_evidence(data.question)

    emb1 = model.encode(data.answer, convert_to_tensor=True)
    emb2 = model.encode(evidence, convert_to_tensor=True)

    score = util.cos_sim(emb1, emb2).item()

    if score > 0.6:
        verdict = "Factual"
    elif score > 0.3:
        verdict = "Uncertain"
    else:
        verdict = "Hallucinated"

    return {
        "answer": data.answer,
        "evidence": evidence,
        "score": round(score, 2),
        "verdict": verdict
    }

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
!uvicorn main:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [4696] using WatchFiles
ERROR:    Error loading ASGI app. Could not import module "main".
INFO:     Stopping reloader process [4696]
